# Is the numeral representation form-invariant? (text encoder only)
The requested count is trivially decodable from the text embedding - but is that *numerical comprehension* or just *lexical numeral identity*? We probe the count from the fused text embedding and test **cross-form transfer**: train on **word** form ('three cats'), test on **digit** form ('3 cats'), and vice versa. High cross-form accuracy = a form-invariant (more abstract) numeral representation; low = lexical only.

**Runtime:** GPU but fast (no image generation).

In [ ]:
import os
if not os.path.exists('src'):
    !git clone https://github.com/serinaqin/T2I-Count-Anomaly.git
    %cd T2I-Count-Anomaly
else:
    !git pull
!pip install -q -r requirements.txt
!pip install -q pytest

In [ ]:
import sys; sys.path.insert(0, '.')
import numpy as np, pandas as pd, os, yaml, torch
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import balanced_accuracy_score
from src.prompts import NUMBER_WORDS, pluralize
from src.pipeline import load_sdxl
from src.config import load_config

In [ ]:
cfg = load_config('configs/exp_seminvariance.yaml')
pipe = load_sdxl()
@torch.no_grad()
def text_embed(prompt):
    # SDXL fused text embedding (mean-pooled over tokens)
    out = pipe.encode_prompt(prompt=prompt, prompt_2=None, device=pipe.device,
                             num_images_per_prompt=1, do_classifier_free_guidance=False)
    return out[0][0].mean(0).detach().cpu().float().numpy()
def word_prompt(n, obj):  return f'{NUMBER_WORDS[n]} {obj if n==1 else pluralize(obj)}'
def digit_prompt(n, obj): return f'{n} {obj if n==1 else pluralize(obj)}'
print('example forms:', word_prompt(3, 'cat'), '|', digit_prompt(3, 'cat'))

In [ ]:
# Build word-form and digit-form embeddings for every (count, object).
Xw, Xd, y = [], [], []
for n in cfg.counts:
    for obj in cfg.objects:
        Xw.append(text_embed(word_prompt(n, obj)))
        Xd.append(text_embed(digit_prompt(n, obj)))
        y.append(n)
Xw, Xd, y = np.array(Xw), np.array(Xd), np.array(y)
print('embeddings:', Xw.shape, '| counts', sorted(set(y)))

In [ ]:
# Within-form (train/test same form, held-out objects) vs cross-form transfer.
def clf():
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=1.0))
def bal(model, X, ytrue):
    return balanced_accuracy_score(ytrue, model.predict(X))
# fit on ALL word, test on ALL digit (and vice versa) = pure form transfer
mw = clf().fit(Xw, y); md = clf().fit(Xd, y)
res = {
  'word->word (train-set fit)': bal(mw, Xw, y),
  'digit->digit (train-set fit)': bal(md, Xd, y),
  'word->DIGIT (cross-form)': bal(mw, Xd, y),
  'digit->WORD (cross-form)': bal(md, Xw, y),
}
for k, v in res.items(): print(f'{k:>28}: balanced acc {v:.3f}')
pd.Series(res).to_csv('results/exp_seminvariance.csv')

In [ ]:
chance = 1.0 / len(set(y))
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(range(len(res)), list(res.values()),
       color=['gray', 'gray', 'C2', 'C2'])
ax.axhline(chance, color='r', ls=':', label=f'chance={chance:.2f}')
ax.set_xticks(range(len(res))); ax.set_xticklabels(list(res.keys()), rotation=20, ha='right')
ax.set_ylabel('balanced accuracy'); ax.set_ylim(0, 1)
ax.set_title('Numeral representation: within-form vs cross-form (word<->digit)')
ax.legend(); plt.tight_layout()
plt.savefig('results/exp_seminvariance.png', dpi=110, bbox_inches='tight'); plt.show()

## How to read this
- **Cross-form (word->digit, digit->word) accuracy well above chance and close to within-form** = the numeral is represented **form-invariantly** in the text embedding -> more than lexical; supports (some) numerical comprehension. We can then keep a measured 'the requested count is faithfully & form-invariantly represented' claim.
- **Cross-form near chance while within-form is high** = the representation is **lexical** (word-token identity), not abstract number -> we downgrade the wording to 'the numeral token is faithfully represented' and drop any 'comprehension' claim (the reviewer's exact point).
- Note: this is the *text* side; it is orthogonal to the causal image-side finding and simply calibrates how strong the comprehension wording may be.